# Лабораторная работа 1. Обработка признаков.

## Задание

1. Выбрать набор данных (датасет), содержащий категориальные и числовые признаки и пропуски в данных. Для выполнения следующих пунктов можно использовать несколько различных наборов данных (один для обработки пропусков, другой для категориальных признаков и т.д.) Просьба не использовать датасет, на котором данная задача решалась в лекции.
2. Для выбранного датасета (датасетов) на основе материалов лекций решить следующие задачи:
    1. устранение пропусков в данных;
    2. кодирование категориальных признаков;
    3. нормализация числовых признаков.

## Выполнение

In [1]:
%pip install pandas numpy seaborn matplotlib scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

### 1. Выбрать набор данных (датасет), содержащий категориальные и числовые признаки и пропуски в данных

Выбран датасет [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data?select=train.csv).

In [3]:
df = pd.read_csv('train.csv')

Колонки с пропусками

In [4]:
missing_info = df.isnull().sum()
print(missing_info[missing_info > 0])

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64


### 2. Устранение пропусков в данных

- Для числовых признаков пропуски заполняются медианным значением
- Для категориальных признаков пропуски заполняются значением `NA`

In [5]:
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

/tmp/ipykernel_11199/441829498.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [6]:
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

In [7]:
cat_imputer = SimpleImputer(strategy='constant', fill_value='NA')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

Пропуски после обработки

In [8]:
missing_info = df.isnull().sum()
print(missing_info[missing_info > 0])

Series([], dtype: int64)


### 3. Кодирование категориальных признаков

Будет кодироваться признак `MSZoning` - тип жилой застройки

In [9]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [10]:
zoning_encoded = ohe.fit_transform(df[['MSZoning']])
zoning_df = pd.DataFrame(zoning_encoded, columns=ohe.get_feature_names_out(['MSZoning']))

In [11]:
df = pd.concat([df, zoning_df], axis=1).drop('MSZoning', axis=1)

In [12]:
print(df.filter(like='MSZoning').head())

   MSZoning_C (all)  MSZoning_FV  MSZoning_RH  MSZoning_RL  MSZoning_RM
0               0.0          0.0          0.0          1.0          0.0
1               0.0          0.0          0.0          1.0          0.0
2               0.0          0.0          0.0          1.0          0.0
3               0.0          0.0          0.0          1.0          0.0
4               0.0          0.0          0.0          1.0          0.0


### 4. Нормализация числовых признаков

In [13]:
scaler = MinMaxScaler()

cols_to_scale = ['GrLivArea', 'LotArea', 'LotFrontage']

df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

print(df[cols_to_scale].head())

   GrLivArea   LotArea  LotFrontage
0   0.259231  0.033420     0.150685
1   0.174830  0.038795     0.202055
2   0.273549  0.046507     0.160959
3   0.260550  0.038561     0.133562
4   0.351168  0.060576     0.215753
